# One-tap equalizer

> Each symbol divided by its channel, which the receiver knows.

In [ ]:
#| default_exp equalization.one_tap

In [ ]:
#| hide
from nbdev.showdoc import *

Equalization undoes the channel: from what the receiver gets, `y`, it estimates the symbols sent, `x_hat`, with
their noise power `no_eff`, for the demapper (see `modulation.mapping`).

`OneTapEqualizer` divides each symbol by its channel, `x̂ = y / h`, which leaves the noise `no / |h|²`. It assumes
the receiver knows `h` (perfect channel state information). It is exact for one antenna, a flat channel per symbol
(one carrier, or OFDM's subcarriers). With several antennas, an LMMSE or zero-forcing equalizer (Sionna's
`sionna.phy.mimo`), and channel estimation from pilots (`sionna.phy.ofdm.LSChannelEstimator`), would be stages
here.

In [ ]:
#| export
from __future__ import annotations

import numpy as np

from comm_core.block import Block, _is_torch
from comm_core.channel.awgn import _per_row

In [ ]:
#| export
class OneTapEqualizer(Block):
    "Each symbol divided by its channel, which the receiver knows: x̂ = y / h, with noise no / |h|²."

    outputs = ('x_hat', 'no_eff')

    def call(
        self,
        y, # What the receiver got, (B, ..., S)
        h_sym, # The channel of each symbol, √P included, broadcast against y
        no, # Each transmission's noise power, (B,)
    ) -> tuple:
        "(x_hat, no_eff), like y."
        if _is_torch(y):
            return y / h_sym, _per_row(no, y) / (h_sym.abs() ** 2).clamp_min(1e-30)
        return y / h_sym, _per_row(no, y) / np.maximum(np.abs(h_sym) ** 2, 1e-30)

In [ ]:
x_hat, no_eff = OneTapEqualizer()(np.array([[2j, -2j]]), np.array([[2j, 1j]]), np.array([0.5]))
assert x_hat.tolist() == [[1, -2]] and no_eff.tolist() == [[0.125, 0.5]]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()